# 07 · SCREEN — deep EHR conditioning (val-only, 1 seed)

**Đây là phép sàng, KHÔNG phải bằng chứng.** Mục đích duy nhất: quyết định có đáng bỏ ra
**70–130 GPU-giờ** cho `PLAN_confirmatory_deep_ehr` hay không.

`06b` đã bác bỏ mạnh **late fusion** trên frozen patch representation (ba phép thử cơ chế
khác nhau, hội tụ). Nhưng nó **chưa** kiểm tra trường hợp EHR can thiệp vào *quá trình hình
thành* representation:

$$Z_l = B_l(Z_{l-1};\, e)$$

Screen này kiểm đúng điều đó, ở dạng rẻ nhất có thể.

## Thiết kế

| | mô tả | vai trò |
|---|---|---|
| **D0** | H5 nguyên trạng (không train thêm) | mốc identity + neo |
| **D1** | adapter **unconditional** trong block 11–12 | **control cùng capacity** |
| **D2** | adapter **EHR-conditioned** trong block 11–12 | model đích |

**Estimand: `D2 − D1`** — không phải `D2 − D0`, vì `D2 − D0` trộn hiệu ứng EHR với
capacity của adapter và với việc train thêm.

**D1 và D2 có số tham số BẰNG NHAU tuyệt đối.** D1 dùng *cùng* `EhrEncoder` nhưng luôn
nhận một input hằng số (labs = 0, mask = 0) thay vì của bệnh nhân. Khác biệt duy nhất giữa
hai nhánh là **context có phụ thuộc bệnh nhân hay không** — không phải capacity.

## Screen này rẻ ở đâu, và trả giá bằng gì

Dùng lại **backbone 04b cũ** và **H5_s42 cũ** (đã train), chỉ train adapter + decoder.
→ ~6h thay vì ~17h/seed của pipeline confirmatory.

Cái phải trả: pipeline này **đã bị nhiễm** — backbone train trên toàn bộ TRAIN cũ, và val
đã từng dùng để chọn model ở 06b. **Vì vậy kết quả ở đây KHÔNG được dùng làm bằng chứng.**
Nó chỉ trả lời một câu hỏi đầu tư: *có tín hiệu đủ để mở pipeline confirmatory không?*

**Chỉ đọc VAL. Không chạm test.** Giữ nguyên khả năng chạy confirmatory sau này.

## Ngưỡng — khóa TRƯỚC khi chạy

| `D2 − D1` trên val | Quyết định |
|---|---|
| `< +0.002` | **Dừng.** Không mở confirmatory. Tiết kiệm ~80–130 GPU-giờ. |
| `+0.002 … +0.004` | Vùng xám — cân nhắc 2–3 seed screen nữa trước khi quyết. |
| `> +0.004` | **Mở confirmatory** theo `PLAN_confirmatory_deep_ehr`. |

Kèm điều kiện cần: `D2_real > D2_shuffle`. Nếu tráo EHR không làm đổi gì thì gain (nếu có)
không đến từ thông tin bệnh nhân.

**Không đổi ngưỡng sau khi xem kết quả.**

## Giới hạn kiến trúc (ghi trước)

γ,β broadcast trên chiều token ⇒ mọi patch nhận **cùng** một điều biến. Screen này kiểm
**patient-specific channel recalibration** ở block ngữ nghĩa. Nó **không** kiểm trực tiếp
việc EHR làm nổi bật một *vùng* cụ thể — hiệu ứng không gian chỉ có thể xuất hiện gián
tiếp qua self-attention của block kế tiếp.

Nếu D2 không hơn D1, kết luận hợp lệ là *"channel-wise conditioning ở hai block cuối không
giúp"*, **không phải** *"mọi deep multimodal fusion đều vô dụng"*.


## C0 · Config + preregistration

In [ ]:
import os, json, math, time, pickle, random, shutil, tarfile
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
OUT_DIR   = os.path.join(DATA_DIR, 'cohort_v2')
SHARD_DIR = os.path.join(DATA_DIR, 'cxr_560_shards')
LOCAL_IMG = '/content/cxr_local'
BB_CKPT   = os.path.join(DATA_DIR, 'lpft_v2', 'raddino_mld_db_ft_best.pt')      # 04b backbone
H5_CKPT   = os.path.join(DATA_DIR, 'token_fusion', 'checkpoints', 'H5_s42_best.pt')
SC_DIR    = os.path.join(DATA_DIR, 'screen_deep_ehr')
os.makedirs(f'{SC_DIR}/checkpoints', exist_ok=True); os.makedirs(f'{SC_DIR}/metrics', exist_ok=True)

# ---- preregistered, do not change after seeing results ----
PREREG = dict(estimand='D2 - D1 (val study-level macro mAP)',
              go_threshold=0.004, gray_zone=(0.002, 0.004), stop_below=0.002,
              require='D2_real > D2_shuffle',
              adapter_blocks=(10, 11),          # 0-indexed -> "block 11,12"
              bottleneck=64, d_e=128,
              ehr_features='7 labs value + 7 labs mask (no micro / vitals / como)',
              seed=42, epochs=12, patience=3, effective_batch=256,
              lr=1e-4, wd=1e-2, warmup_frac=0.05, grad_clip=5.0, loss='ASL',
              evaluate_on='val ONLY -- test/holdout untouched')
json.dump(PREREG, open(f'{SC_DIR}/preregistration.json','w'), indent=1)
print(json.dumps(PREREG, indent=1))

W_F, W_L = 0.7, 0.3
SEED = PREREG['seed']
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('\ndevice:', dev, '|', torch.cuda.get_device_name(0) if dev=='cuda' else 'CPU')
for p in (BB_CKPT, H5_CKPT):
    assert os.path.exists(p), f'{p} not found'

## C0-bis · Stage ảnh về ổ local

`/content` bị xóa mỗi lần VM restart, nên cell này luôn phải chạy trong phiên hiện tại.
Resume bằng marker `.done_`; tức thì nếu ổ còn ấm. **CPU, không cần GPU** (~20 phút nếu VM
mới — Drive đơn luồng ~13 MB/s).

In [ ]:
os.makedirs(LOCAL_IMG, exist_ok=True)
tars = sorted(f for f in os.listdir(SHARD_DIR) if f.endswith('.tar'))
assert tars, f'no shards in {SHARD_DIR}'
todo = [f for f in tars if not os.path.exists(os.path.join(LOCAL_IMG, f'.done_{f}'))]
print(f'shards: {len(tars)} on Drive | {len(todo)} to stage')
for f in tqdm(todo, desc='staging shards', unit='shard'):
    tmp = f'/content/{f}'
    shutil.copyfile(os.path.join(SHARD_DIR, f), tmp)
    with tarfile.open(tmp) as t: t.extractall(LOCAL_IMG)
    os.remove(tmp)
    open(os.path.join(LOCAL_IMG, f'.done_{f}'), 'w').close()
n_local = len([f for f in os.listdir(LOCAL_IMG) if f.endswith('.jpg')])
print(f'local images: {n_local:,}'); assert n_local > 0

## C1 · Cohort + **chỉ labs** (§6.4 của plan)

Chỉ dùng nhóm feature đã có bằng chứng mạnh nhất: **7 labs value + 7 mask**. Không dùng
microbiology / vitals / comorbidity — cả ba đã cho gain ≈ 0 trong ablation `06`, nên thêm
vào chỉ tăng capacity và nhiễu, làm khó diễn giải một kết quả null.

In [ ]:
meta   = json.load(open(os.path.join(OUT_DIR, 'cohort_meta.json')))
LABELS = meta['labels']; assert len(LABELS) == 13
ehr = pd.read_parquet(os.path.join(OUT_DIR, 'cohort_study_level.parquet')).sort_values('study_id').reset_index(drop=True)
for c in ('study_id','subject_id','split','setting'):
    assert c in ehr.columns, f'ehr has no {c}; cols={sorted(ehr.columns)}'
N = len(ehr)
vocab = pickle.load(open(os.path.join(OUT_DIR, 'ehr_vocab.pkl'), 'rb'))
LABS = vocab['labs']
lab_val  = np.stack([ehr[f'{c}_value'].to_numpy('float32') for c in LABS], 1)
lab_mask = np.stack([ehr[f'{c}_mask'].to_numpy('float32')  for c in LABS], 1)
D_EHR_IN = 2*len(LABS)
Y_STUDY  = ehr[LABELS].to_numpy('float32')
EHR_ROW  = {int(s): i for i, s in enumerate(ehr.study_id.to_numpy())}
print(f'cohort {N:,} | labs {len(LABS)} -> EHR input dim {D_EHR_IN}')

# ---- image manifest (same construction as 06b C3) ----
vm = pd.read_parquet(os.path.join(OUT_DIR, 'view_map.parquet'))
vm = vm[vm.view_group.isin(['frontal','lateral'])]
_dup = [c for c in ('subject_id','split','setting') if c in vm.columns]   # avoid _x/_y collision
man = vm.drop(columns=_dup).merge(ehr[['study_id','subject_id','split']], on='study_id', how='inner')
before = len(man)
man = man[man.dicom_id.map(lambda d: os.path.exists(f'{LOCAL_IMG}/{d}.jpg'))].sort_values('dicom_id').reset_index(drop=True)
assert (before-len(man))/before < 0.01, f'{before-len(man)} images missing -> run C0-bis'
man['ehr_row'] = man.study_id.map(EHR_ROW).astype('int64')
IDX = {s: np.where(man.split.to_numpy()==s)[0] for s in ('train','val','test')}
print('images by split:', {s: len(IDX[s]) for s in IDX})
print('NOTE: test index built but NEVER used in this notebook.')

## C2 · Dataset ở mức ảnh (giống 06b)

`num_workers` cho train, **0 cho predict** — loader dựng lại mỗi epoch với worker>0 sẽ ngập
`can only test a child process` (vô hại nhưng nuốt mất output).

In [ ]:
from transformers import AutoImageProcessor
from torchvision import transforms as T
from PIL import Image
proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MODEL_SIZE = proc.crop_size['height']; MEAN, STD = proc.image_mean[0], proc.image_std[0]
train_tf = T.Compose([T.RandomResizedCrop(MODEL_SIZE, scale=(0.85,1.0), ratio=(0.95,1.05),
                                          interpolation=T.InterpolationMode.BICUBIC),
                      T.RandomHorizontalFlip(p=0.0), T.RandomRotation(7),
                      T.ToTensor(), T.Normalize([MEAN]*3,[STD]*3)])
eval_tf  = T.Compose([T.CenterCrop(MODEL_SIZE), T.ToTensor(), T.Normalize([MEAN]*3,[STD]*3)])

class ImgDS(Dataset):
    def __init__(self, idx, tf, perm=None): self.idx, self.tf, self.perm = np.asarray(idx), tf, perm
    def __len__(self): return len(self.idx)
    def __getitem__(self, k):
        i = self.idx[k]; r = man.iloc[i]
        im = Image.open(f'{LOCAL_IMG}/{r.dicom_id}.jpg').convert('L')
        im = Image.merge('RGB', (im, im, im))
        tr = int(r.ehr_row)                       # TRUE row -> labels always from here
        er = tr if self.perm is None else int(self.perm[tr])
        return dict(image=self.tf(im), y=torch.from_numpy(Y_STUDY[tr]), row=i,
                    lab_val=torch.from_numpy(lab_val[er]), lab_mask=torch.from_numpy(lab_mask[er]))

def collate(b):
    return dict(image=torch.stack([x['image'] for x in b]), y=torch.stack([x['y'] for x in b]),
                row=torch.tensor([x['row'] for x in b]),
                lab_val=torch.stack([x['lab_val'] for x in b]),
                lab_mask=torch.stack([x['lab_mask'] for x in b]))

def loader(split, tf, shuffle, bs, perm=None, workers=8):
    return DataLoader(ImgDS(IDX[split], tf, perm), batch_size=bs, shuffle=shuffle,
                      collate_fn=collate, num_workers=workers, pin_memory=(dev=='cuda'),
                      persistent_workers=(workers>0))
to_dev = lambda b: {k: v.to(dev, non_blocking=True) for k, v in b.items()}
print('loaders ready')

## C3 · Backbone + **xác minh cấu trúc block** (không giả định)

Adapter phải chèn *bên trong* backbone, nên ta phải tự chạy vòng lặp qua các block thay vì
gọi `backbone(x)`. Điều đó chỉ an toàn nếu vòng lặp thủ công **tái tạo đúng** output gốc —
nên đây là một assert, không phải một niềm tin:

```
manual_forward(x)  ==  backbone(x).last_hidden_state
```

Nếu HF đổi cấu trúc `Dinov2` (thứ tự layernorm, layer trả tuple hay tensor…), cell chết ở
đây chứ không train ra một model sai âm thầm.

In [ ]:
from transformers import AutoModel
backbone = AutoModel.from_pretrained('microsoft/rad-dino').to(dev).eval()
_sd = torch.load(BB_CKPT, map_location=dev, weights_only=False)['model']
_bb = {k[len('backbone.'):]: v for k, v in _sd.items() if k.startswith('backbone.')}
assert _bb; backbone.load_state_dict(_bb, strict=True)
for p in backbone.parameters(): p.requires_grad = False
D = backbone.config.hidden_size; assert D == 768

LAYERS = backbone.encoder.layer
n_blocks = len(LAYERS)
print(f'blocks: {n_blocks} | hidden {D}')
print('module path:', type(backbone).__name__, '-> encoder.layer[i] ->', type(LAYERS[0]).__name__)
ADAPTER_BLOCKS = tuple(PREREG['adapter_blocks'])
SHARED_UPTO    = min(ADAPTER_BLOCKS)          # blocks [0, SHARED_UPTO) chạy chung, no_grad
assert max(ADAPTER_BLOCKS) < n_blocks
print(f'shared (frozen, no_grad): blocks 0..{SHARED_UPTO-1}  |  adapters at blocks {ADAPTER_BLOCKS}')

def _layer(h, i):
    o = LAYERS[i](h)
    return o[0] if isinstance(o, (tuple, list)) else o

def forward_shared(x):
    """Blocks [0, SHARED_UPTO). Chạy MỘT lần, dùng chung cho D0/D1/D2."""
    with torch.no_grad():
        h = backbone.embeddings(x)
        for i in range(SHARED_UPTO): h = _layer(h, i)
    return h.detach()

def forward_tail(h, adapters=None, e=None):
    """Blocks [SHARED_UPTO, n). Adapter chèn SAU mỗi block được chỉ định."""
    for i in range(SHARED_UPTO, n_blocks):
        h = _layer(h, i)
        if adapters is not None and str(i) in adapters:
            h = adapters[str(i)](h, e)
    return backbone.layernorm(h)

# ---- THE gate: manual loop must reproduce the stock forward exactly ----
_x = torch.randn(2, 3, MODEL_SIZE, MODEL_SIZE, device=dev)
with torch.no_grad():
    ref = backbone(pixel_values=_x).last_hidden_state
    got = forward_tail(forward_shared(_x))
md_ = (ref-got).abs().max().item()
assert torch.allclose(ref, got, atol=1e-4), f'manual forward != stock forward (max diff {md_:.2e})'
print(f'G0 manual forward == backbone(x).last_hidden_state   max|diff| {md_:.2e}   OK')
print(f'   shapes: {tuple(ref.shape)}  (B, 1+patches, {D})')

## C4 · EHR encoder (labs-only) + conditional adapter

$$u = W_{down}\,\mathrm{LN}(Z)\quad
(\gamma,\beta)=G(e)\quad
\tilde u=(1+\gamma)\odot u+\beta\quad
\Delta Z = W_{up}\,\mathrm{GELU}(\tilde u)\quad
Z' = Z+\Delta Z$$

**`W_up` zero-init ⇒ `ΔZ = 0` ⇒ identity TUYỆT ĐỐI ở step 0.** Đây là điểm mạnh hơn hẳn
so với EHR token của `06b`: chèn thêm một token — kể cả token rỗng — vẫn đổi mẫu số softmax
nên `H6z ≠ H5`. Adapter residual thì không: nó cộng đúng số 0.

Chỉ `W_up` được zero-init. `W_down` và `G` giữ init chuẩn, nếu không gradient sẽ chết ở cả
đường lên lẫn đường xuống.

In [ ]:
class EhrEncoderLabs(nn.Module):
    """labs value+mask -> e[128]. D1 và D2 dùng CÙNG lớp này (param-matched tuyệt đối);
    khác biệt duy nhất là D1 luôn nhận input hằng số."""
    def __init__(self, d_in=D_EHR_IN, d_e=PREREG['d_e'], p=0.2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d_e), nn.LayerNorm(d_e), nn.GELU(), nn.Dropout(p),
                                 nn.Linear(d_e, d_e), nn.LayerNorm(d_e), nn.GELU())
    def forward(self, b):
        return self.net(torch.cat([b['lab_val'], b['lab_mask']], 1))

class CondAdapter(nn.Module):
    def __init__(self, d=768, r=PREREG['bottleneck'], d_e=PREREG['d_e']):
        super().__init__()
        self.ln   = nn.LayerNorm(d)
        self.down = nn.Linear(d, r)
        self.gen  = nn.Linear(d_e, 2*r)                 # -> gamma, beta (broadcast over tokens)
        self.up   = nn.Linear(r, d)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)   # identity at step 0
        nn.init.zeros_(self.gen.weight); nn.init.zeros_(self.gen.bias) # start at gamma=0,beta=0
    def forward(self, Z, e):
        u = self.down(self.ln(Z))                       # [B,N,r]
        g, b = self.gen(e).chunk(2, -1)                 # [B,r] each
        u = (1 + g[:, None, :]) * u + b[:, None, :]     # broadcast over N tokens
        return Z + self.up(F.gelu(u))

def make_adapters():
    return nn.ModuleDict({str(i): CondAdapter() for i in ADAPTER_BLOCKS})
print('adapter params/block:', sum(p.numel() for p in CondAdapter().parameters()))
print('ehr encoder params  :', sum(p.numel() for p in EhrEncoderLabs().parameters()))

## C5 · D0 / D1 / D2 — warm-start `norm` + ML-Decoder từ **H5_s42**

Warm-start là **bắt buộc** để assert identity có nghĩa: nếu D1/D2 khởi tạo decoder mới còn
D0 dùng decoder đã train của H5 thì logits không thể bằng nhau dù adapter đúng.

`D1` và `D2` khác nhau **đúng một chỗ**: `D2` đưa EHR của bệnh nhân vào encoder, `D1` đưa
một vector hằng (labs=0, mask=0). Cùng lớp, cùng số tham số, cùng optimizer, cùng LR.

In [ ]:
import sys, types
ML_SRC = '/content/ml_decoder.py'
if not os.path.exists(ML_SRC):
    !wget -q -O {ML_SRC} https://raw.githubusercontent.com/Alibaba-MIIL/ML_Decoder/main/src_files/ml_decoder/ml_decoder.py
sys.path.insert(0, '/content')
import ml_decoder as _mld
from ml_decoder import MLDecoder
if not hasattr(_mld.TransformerDecoderLayerOptimal, '_patched'):
    _oi = _mld.TransformerDecoderLayerOptimal.__init__
    def _init(self, *a, **kw):
        _oi(self, *a, **kw); self.self_attn = types.SimpleNamespace(batch_first=False)
    _mld.TransformerDecoderLayerOptimal.__init__ = _init
    _of = _mld.TransformerDecoderLayerOptimal.forward
    def _fwd(self, tgt, memory, **kw):
        kw.pop('tgt_is_causal', None); kw.pop('memory_is_causal', None)
        return _of(self, tgt, memory, **kw)
    _mld.TransformerDecoderLayerOptimal.forward = _fwd
    _mld.TransformerDecoderLayerOptimal._patched = True

H5_SD = torch.load(H5_CKPT, map_location=dev, weights_only=False)['model']
print('H5 checkpoint keys (head):', [k for k in H5_SD if not k.startswith('mld.')][:6])

class DModel(nn.Module):
    """mode: 'plain' (D0) | 'uncond' (D1) | 'cond' (D2)"""
    def __init__(self, mode):
        super().__init__(); self.mode = mode
        self.norm = nn.LayerNorm(768)                      # H5's patch LayerNorm
        self.mld  = MLDecoder(num_classes=len(LABELS), initial_num_features=768,
                              decoder_embedding=768, num_of_groups=len(LABELS))
        if mode != 'plain':
            self.enc = EhrEncoderLabs()
            self.adapters = make_adapters()
        # warm-start from H5 so the step-0 identity assert is meaningful
        self.norm.load_state_dict({k.split('norm.')[1]: v for k, v in H5_SD.items()
                                   if k.startswith('norm.')})
        self.mld.load_state_dict({k.split('mld.')[1]: v for k, v in H5_SD.items()
                                  if k.startswith('mld.')})
    def forward(self, z_shared, batch):
        if self.mode == 'plain':
            h = forward_tail(z_shared)
        else:
            if self.mode == 'cond':
                e = self.enc(batch)
            else:                                          # constant context, same params
                z = torch.zeros(batch['lab_val'].shape[0], len(LABS), device=z_shared.device)
                e = self.enc({'lab_val': z, 'lab_mask': z})
            h = forward_tail(z_shared, self.adapters, e)
        return self.mld(self.norm(h[:, 1:]))               # patches only, like H5

torch.manual_seed(SEED)
D0 = DModel('plain').to(dev).eval()
D1 = DModel('uncond').to(dev)
D2 = DModel('cond').to(dev)
D2.adapters.load_state_dict(D1.adapters.state_dict())      # same adapter init
D2.enc.load_state_dict(D1.enc.state_dict())                # same encoder init
n = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f'\ntrainable: D1 {n(D1)/1e6:.3f} M | D2 {n(D2)/1e6:.3f} M | '
      f'rel.diff {abs(n(D1)-n(D2))/max(n(D1),n(D2)):.4%}  (yêu cầu < 5%)')
assert abs(n(D1)-n(D2))/max(n(D1),n(D2)) < 0.05

## C6 · Cổng kiểm tra — chạy TRƯỚC khi tốn GPU

1. **Identity ở logit level** (fp32, eval, dropout tắt): `D1 == D0` và `D2 == D0` tại step 0.
2. **Gradient routing**: backbone sạch; adapter/encoder/decoder của mỗi nhánh có gradient;
   nhánh kia **không** dính gradient của nhánh này.
3. **D2 thật sự đọc EHR**: đổi labs → `e` phải đổi. Nếu không, mọi kết quả sau vô nghĩa.

In [ ]:
b = to_dev(collate([ImgDS(IDX['train'], eval_tf)[k] for k in range(8)]))
for m in (D0, D1, D2): m.eval()
with torch.no_grad(), torch.autocast('cuda', enabled=False):
    z = forward_shared(b['image'].float())
    y0, y1, y2 = D0(z, b), D1(z, b), D2(z, b)
d1, d2 = (y0-y1).abs().max().item(), (y0-y2).abs().max().item()
print(f'G1 identity  max|D0-D1| {d1:.2e} | max|D0-D2| {d2:.2e}')
assert d1 < 1e-4 and d2 < 1e-4, 'identity init broken -> D2-D1 would be uninterpretable'
print('   OK  (adapter residual cộng đúng 0 -- identity TUYỆT ĐỐI, khác token của 06b)')

# G2 -- D2 phải phản ứng với EHR
with torch.no_grad():
    e_real = D2.enc(b)
    e_flip = D2.enc({'lab_val': b['lab_val'].flip(0), 'lab_mask': b['lab_mask'].flip(0)})
    de = (e_real - e_flip).abs().mean().item()
print(f'G2 d(e) khi đổi bệnh nhân: {de:.4f}')
assert de > 1e-3, 'EHR encoder không phản ứng với input -> kiểm tra featurization'

# G3 -- gradient routing
for m in (D1, D2): m.train(); m.zero_grad(set_to_none=True)
crit_probe = lambda s, y: F.binary_cross_entropy_with_logits(s, y)
z = forward_shared(b['image'])
crit_probe(D2(z, b), b['y']).backward()
assert all(p.grad is None for p in backbone.parameters()), 'GRADIENT LEAKED INTO BACKBONE'
assert any(p.grad is not None for p in D2.adapters.parameters())
assert any(p.grad is not None for p in D2.enc.parameters())
assert all(p.grad is None for p in D1.parameters()), 'D1 nhận gradient của D2 -> hai nhánh dùng chung tham số!'
print('G3 gradients: backbone sạch · D2 có grad · D1 không dính  OK')
for m in (D1, D2): m.zero_grad(set_to_none=True)

## C7 · ASL + pooling study-level 7:3

In [ ]:
from sklearn.metrics import average_precision_score

class ASLoss(nn.Module):
    """Alibaba-MIIL/ASL, math verbatim."""
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        super().__init__(); self.gn, self.gp, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps
    def forward(self, x, y):
        y = y.float(); xs_pos = torch.sigmoid(x); xs_neg = 1.0 - xs_pos
        if self.clip: xs_neg = (xs_neg + self.clip).clamp(max=1)
        los = y*torch.log(xs_pos.clamp(min=self.eps)) + (1-y)*torch.log(xs_neg.clamp(min=self.eps))
        pt = xs_pos*y + xs_neg*(1-y); g = self.gp*y + self.gn*(1-y)
        return -(los*torch.pow(1-pt, g)).mean()
with torch.no_grad():
    _a = ASLoss(); _e = _a(torch.tensor([[-4.6]]), torch.tensor([[0.]])).item()
    _h = _a(torch.tensor([[4.6]]),  torch.tensor([[0.]])).item()
assert _e < 1e-3 and _h > 1.0
print(f'ASL check: easy-neg {_e:.5f} | hard-neg {_h:.3f}  OK')

@torch.no_grad()
def predict(models, split, bs, perm=None):
    for m in models.values(): m.eval()
    P = {t: np.zeros((len(IDX[split]), len(LABELS)), 'float32') for t in models}
    pos = 0
    for b in tqdm(loader(split, eval_tf, False, bs, perm, workers=0), desc=f'pred[{split}]', leave=False):
        b = to_dev(b); z = forward_shared(b['image']); k = z.shape[0]
        for t, m in models.items():
            P[t][pos:pos+k] = torch.sigmoid(m(z, b)).float().cpu().numpy()
        pos += k
    return P

def pool_study(p_img, split):
    sub = man.iloc[IDX[split]]
    d = pd.DataFrame(p_img, columns=LABELS)
    d['study_id'] = sub.study_id.to_numpy(); d['vg'] = sub.view_group.to_numpy()
    sid = np.sort(d.study_id.unique())
    fr = d[d.vg=='frontal'].groupby('study_id')[LABELS].mean().reindex(sid)
    la = d[d.vg=='lateral'].groupby('study_id')[LABELS].mean().reindex(sid)
    hf = fr.notna().all(1).to_numpy().astype('float32'); hl = la.notna().all(1).to_numpy().astype('float32')
    den = (W_F*hf + W_L*hl)[:, None]; assert (den > 0).all()
    return sid, (W_F*np.nan_to_num(fr.to_numpy('float32'))*hf[:,None]
                 + W_L*np.nan_to_num(la.to_numpy('float32'))*hl[:,None]) / den

def macro_mAP(p_img, split):
    sid, PS = pool_study(p_img, split)
    Yt = ehr.set_index('study_id').loc[sid, LABELS].to_numpy('float32')
    return float(np.mean([average_precision_score(Yt[:,j], PS[:,j]) for j in range(len(LABELS))]))

## C8 · Stage A — train D1 và D2 trên **một** forward chung

Blocks 0–9 chạy `no_grad` **một lần**, dùng chung cho cả hai nhánh (common random numbers →
cùng ảnh, cùng augmentation, cùng thứ tự). Blocks 10–11 phải chạy riêng cho mỗi nhánh vì
adapter nằm bên trong.

Backward **tuần tự**, mỗi nhánh optimizer/scheduler/clip **riêng**. Clip global sẽ scale
gradient của cả hai theo một norm chung ⇒ nhánh mạnh bóp nhánh yếu ⇒ mất tính tương đương
với việc train riêng.

In [ ]:
def probe_batch(cands=(16,24,32,48,64)):
    if dev != 'cuda': return 8
    ok = cands[0]
    for B in cands:
        try:
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            bb = to_dev(collate([ImgDS(IDX['train'], train_tf)[k] for k in range(B)]))
            z = forward_shared(bb['image'])
            for m in (D1, D2): ASLoss()(m(z, bb), bb['y']).backward()
            pk = torch.cuda.max_memory_allocated()/1e9; tot = torch.cuda.get_device_properties(0).total_memory/1e9
            print(f'  B={B:3d} peak {pk:5.1f}/{tot:.0f} GB ({pk/tot:5.1%})')
            if pk/tot < 0.80: ok = B
            else: break
        except torch.cuda.OutOfMemoryError:
            print(f'  B={B:3d} OOM'); break
        finally:
            for m in (D1, D2): m.zero_grad(set_to_none=True)
    torch.cuda.empty_cache(); return ok

def pgroups(m, wd):
    dec = [p for p in m.parameters() if p.requires_grad and p.ndim > 1]
    nod = [p for p in m.parameters() if p.requires_grad and p.ndim <= 1]
    return [{'params':dec,'weight_decay':wd},{'params':nod,'weight_decay':0.0}]

CK = f'{SC_DIR}/checkpoints/_resume_s{SEED}.pt'
def train_screen(batch, epochs=PREREG['epochs']):
    dl = loader('train', train_tf, True, batch)
    accum = max(1, PREREG['effective_batch']//batch)
    total = epochs*math.ceil(len(dl)/accum)
    crit = ASLoss()
    st = {}
    for tag, m in (('D1',D1), ('D2',D2)):
        opt = torch.optim.AdamW(pgroups(m, PREREG['wd']), lr=PREREG['lr'])
        f = lambda s, T=total, W=int(PREREG['warmup_frac']*total): \
            s/max(W,1) if s < W else 0.5*(1+math.cos(math.pi*(s-W)/max(T-W,1)))
        st[tag] = dict(m=m, opt=opt, sch=torch.optim.lr_scheduler.LambdaLR(opt, f),
                       best=-1.0, since=0, done=False, best_sd=None)
    start = 0
    if os.path.exists(CK):
        c = torch.load(CK, map_location=dev, weights_only=False)
        if c.get('batch') != batch:
            raise RuntimeError(f"resume dùng batch={c.get('batch')} nhưng đang chạy batch={batch}; "
                               f"LR schedule sẽ sai. Xóa {CK} hoặc dùng lại batch cũ.")
        for t in st:
            a = c['arms'][t]; st[t]['m'].load_state_dict(a['m']); st[t]['opt'].load_state_dict(a['opt'])
            st[t]['sch'].load_state_dict(a['sch'])
            st[t].update(best=a['best'], since=a['since'], done=a['done'], best_sd=a['best_sd'])
        start = c['epoch']+1
        print(f"RESUMED ep{start}/{epochs}  " + ' '.join(f"{t}:{st[t]['best']:.4f}" for t in st))
    if start >= epochs or all(s['done'] for s in st.values()):
        for t in st: st[t]['m'].load_state_dict(st[t]['best_sd'])
        return {t: st[t]['best'] for t in st}
    print(f'batch {batch} x accum {accum} = {batch*accum} effective | {len(dl)} it/ep | {total} steps')
    for ep in range(start, epochs):
        for t in st: st[t]['m'].train()
        for i, b in enumerate(tqdm(dl, desc=f'ep{ep+1}/{epochs}', leave=False)):
            b = to_dev(b)
            z = forward_shared(b['image'])                 # THE shared forward (blocks 0..9)
            for t, s in st.items():
                if s['done']: continue
                (crit(s['m'](z, b), b['y'])/accum).backward()
                if (i+1) % accum == 0 or i == len(dl)-1:
                    nn.utils.clip_grad_norm_(s['m'].parameters(), PREREG['grad_clip'])  # PER ARM
                    s['opt'].step(); s['sch'].step(); s['opt'].zero_grad(set_to_none=True)
        Pv = predict({t: s['m'] for t, s in st.items() if not s['done']}, 'val', batch*2)
        line = []
        for t, s in st.items():
            if s['done']: line.append(f'{t}:done'); continue
            v = macro_mAP(Pv[t], 'val')
            if v > s['best']:
                s['best'], s['since'] = v, 0
                s['best_sd'] = {k: x.detach().cpu().clone() for k, x in s['m'].state_dict().items()}
            else:
                s['since'] += 1
                if s['since'] >= PREREG['patience']: s['done'] = True
            line.append(f'{t}:{v:.4f}{"*" if s["since"]==0 else ""}')
        torch.save({'epoch':ep,'batch':batch,'arms':{t:dict(m=s['m'].state_dict(),opt=s['opt'].state_dict(),
                    sch=s['sch'].state_dict(),best=s['best'],since=s['since'],done=s['done'],
                    best_sd=s['best_sd']) for t,s in st.items()}}, CK+'.tmp')
        os.replace(CK+'.tmp', CK)
        print(f'  ep{ep+1} val ' + ' '.join(line) + f"  D2-D1 {st['D2']['best']-st['D1']['best']:+.4f}  [saved]")
        if all(s['done'] for s in st.values()): print('  cả hai early-stop'); break
    for t in st: st[t]['m'].load_state_dict(st[t]['best_sd'])
    return {t: st[t]['best'] for t in st}

print('probing batch...'); BATCH = probe_batch(); print('-> BATCH =', BATCH)
t0 = time.time(); BEST = train_screen(BATCH)
print(f'\nscreen done in {(time.time()-t0)/60:.1f} min')

## C9 · Đọc kết quả — so với ngưỡng đã khóa ở C0

`D0` được chấm một lần làm neo (nó chính là H5, không train thêm). Estimand là **`D2 − D1`**.
Kèm chẩn đoán nhân quả: tráo EHR ở mức study, trong cùng split — nếu `real ≈ shuffle` thì
gain (nếu có) không đến từ thông tin bệnh nhân.

In [ ]:
rng = np.random.default_rng(0)
perm = np.arange(N)
key = ehr.split.astype(str) + '|' + ehr.setting.astype(str).str.lower()
for k in key.unique():
    idx = np.where(key.to_numpy()==k)[0]; perm[idx] = rng.permutation(idx)
print(f'shuffle: {(perm!=np.arange(N)).mean():.1%} study đổi EHR row (trong cùng split × setting)')

P  = predict({'D0':D0, 'D1':D1, 'D2':D2}, 'val', BATCH*2)
Ps = predict({'D2':D2}, 'val', BATCH*2, perm=perm)
res = {t: macro_mAP(P[t], 'val') for t in P}
res['D2_shuffle'] = macro_mAP(Ps['D2'], 'val')

print('\n===== VAL macro mAP (1 seed — SÀNG LỌC, không phải bằng chứng) =====')
for t in ('D0','D1','D2','D2_shuffle'): print(f'  {t:11s} {res[t]:.4f}')
delta   = res['D2'] - res['D1']
cap     = res['D1'] - res['D0']
shuf_d  = res['D2'] - res['D2_shuffle']
print(f'\n  D1 - D0  (capacity của adapter)      {cap:+.4f}')
print(f'  D2 - D1  (ESTIMAND: hiệu ứng EHR)    {delta:+.4f}')
print(f'  D2_real - D2_shuffle (nhân quả)      {shuf_d:+.4f}')

go, gray = PREREG['go_threshold'], PREREG['gray_zone']
print('\n===== QUYẾT ĐỊNH (ngưỡng khóa trước, C0) =====')
if delta < gray[0]:
    print(f'  D2-D1 = {delta:+.4f} < +{gray[0]:.3f}  ->  DỪNG.')
    print('  Không mở confirmatory. Tiết kiệm ~80-130 GPU-giờ.')
elif delta < go:
    print(f'  D2-D1 = {delta:+.4f} nằm trong vùng xám [{gray[0]:.3f}, {go:.3f})')
    print('  -> chạy thêm 2-3 seed screen trước khi quyết.')
else:
    print(f'  D2-D1 = {delta:+.4f} > +{go:.3f}  ->  MỞ CONFIRMATORY.')
if shuf_d <= 0 and delta >= gray[0]:
    print('  !! nhưng real <= shuffle: gain KHÔNG đến từ thông tin bệnh nhân -> coi như no-go.')

json.dump(dict(**res, delta_D2_D1=delta, adapter_capacity=cap, shuffle_delta=shuf_d,
               prereg=PREREG), open(f'{SC_DIR}/metrics/screen_result.json','w'), indent=1)
print(f'\nsaved -> {SC_DIR}/metrics/screen_result.json')
print('NHẮC: test/holdout chưa bị chạm. Phương án confirmatory vẫn nguyên vẹn.')